# Experiment: SpectralBio Failure-Mode Packaging (T4)

This notebook packages the screen and validation outputs into manuscript-ready artifacts without changing the underlying benchmark contract.

## Why this notebook exists
- The scientific value only matters if we can express it cleanly in the paper.
- This notebook turns notebooks 7 and 8 into figures, a submission-readiness memo, and compact text inserts for `abstract` and `content`.

## Success criteria
- One zip bundle with figures, tables, and markdown snippets.
- A clear readiness memo saying whether the new result is strong enough for the abstract or only for the content.


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = False
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_OUTPUT_SUBDIR = Path("MyDrive/SpectralBioFailureModePackaging")
SAVE_RESULTS_ZIP = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount(str(DRIVE_MOUNT_POINT))
    print(f"Drive mounted at {DRIVE_MOUNT_POINT}")
else:
    print("Drive mount skipped; outputs stay in the runtime unless OUTPUT_ROOT is changed later.")

print("ACABEI PODE IR PARA O PR?XIMO")


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("SPECTRALBIO_REPO_URL", "https://github.com/DaviBonetto/SpectralBio.git")
REPO_BRANCH = os.environ.get("SPECTRALBIO_REPO_BRANCH", "codex/claw4s-rebuild")
DEFAULT_REPO_ROOT = Path("/content/Stanford-Claw4s")
ENV_REPO_ROOT = os.environ.get("SPECTRALBIO_REPO_ROOT", "").strip()

def _looks_like_repo(path: Path) -> bool:
    return (path / "src" / "spectralbio").exists() and (path / "notebooks").exists()

candidate_roots = []
if ENV_REPO_ROOT:
    candidate_roots.append(Path(ENV_REPO_ROOT).expanduser())
candidate_roots.extend([Path.cwd(), Path.cwd().parent, DEFAULT_REPO_ROOT])
REPO_ROOT = next((path.resolve() for path in candidate_roots if _looks_like_repo(path)), DEFAULT_REPO_ROOT)
BOOTSTRAP_MARKER = REPO_ROOT / ".colab_bootstrap_failure_mode_packaging_t4_complete"

if not _looks_like_repo(REPO_ROOT):
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)])

os.chdir(REPO_ROOT)
subprocess.run(["git", "fetch", "origin", REPO_BRANCH], check=False)
subprocess.run(["git", "checkout", REPO_BRANCH], check=False)

src_path = REPO_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

if not BOOTSTRAP_MARKER.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "pandas==2.2.3", "matplotlib==3.9.2"])
    BOOTSTRAP_MARKER.write_text("ok\n", encoding="utf-8")
    print("Dependencies installed without restarting the runtime.")
else:
    print("Bootstrap marker found; skipping reinstall.")

print("REPO_ROOT =", REPO_ROOT)
print("Python =", sys.version.split()[0])
print("ACABEI PODE IR PARA O PR?XIMO")


## Plan

1. Load the outputs of notebooks 7 and 8.
2. Collect the strongest figure/table artifacts into one place.
3. Write short text inserts and a readiness memo.
4. Export one final zip bundle.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import FileLink, display

from spectralbio.utils.io import ensure_dir

OUTPUT_ROOT = (
    DRIVE_MOUNT_POINT / DRIVE_OUTPUT_SUBDIR
    if USE_GOOGLE_DRIVE
    else REPO_ROOT / "supplementary" / "failure_mode_packaging_t4"
)
ARTIFACT_ROOT = OUTPUT_ROOT / "failure_mode_packaging"
TABLES_DIR = ARTIFACT_ROOT / "tables"
FIGURES_DIR = ARTIFACT_ROOT / "figures"
TEXT_DIR = ARTIFACT_ROOT / "text"

for directory in (ARTIFACT_ROOT, TABLES_DIR, FIGURES_DIR, TEXT_DIR):
    ensure_dir(directory)

def resolve_existing_path(raw: str | Path) -> Path:
    raw_path = Path(raw)
    if raw_path.exists():
        return raw_path
    raw_text = str(raw).replace("\\", "/")
    for prefix in (
        "/content/Stanford-Claw4s/",
        "/teamspace/studios/this_studio/Stanford-Claw4s/",
    ):
        if raw_text.startswith(prefix):
            candidate = REPO_ROOT / raw_text[len(prefix):]
            if candidate.exists():
                return candidate
    return raw_path

SCREEN_SUMMARY_PATH_OVERRIDE = os.environ.get("SPECTRALBIO_FAILURE_SCREEN_SUMMARY", "").strip()
VALIDATION_SUMMARY_PATH_OVERRIDE = os.environ.get("SPECTRALBIO_FAILURE_VALIDATION_SUMMARY", "").strip()

screen_summary_candidates = []
if SCREEN_SUMMARY_PATH_OVERRIDE:
    screen_summary_candidates.append(Path(SCREEN_SUMMARY_PATH_OVERRIDE))
screen_summary_candidates.append(
    REPO_ROOT / "supplementary" / "failure_mode_screen_t4" / "failure_mode_screen" / "tables" / "candidate_failure_mode_summary.json"
)
if USE_GOOGLE_DRIVE:
    screen_summary_candidates.append(
        DRIVE_MOUNT_POINT / Path("MyDrive/SpectralBioFailureModeScreen/failure_mode_screen/tables/candidate_failure_mode_summary.json")
    )
screen_summary_candidates += sorted(REPO_ROOT.glob("supplementary/**/candidate_failure_mode_summary.json"))
screen_summary_path = None
for candidate in screen_summary_candidates:
    resolved = resolve_existing_path(candidate)
    if resolved.exists():
        screen_summary_path = resolved
        break
if screen_summary_path is None or not screen_summary_path.exists():
    raise FileNotFoundError("Could not find candidate_failure_mode_summary.json. Run notebook 7 first and keep its outputs available.")
screen_summary = json.loads(screen_summary_path.read_text(encoding="utf-8"))

validation_summary_candidates = []
if VALIDATION_SUMMARY_PATH_OVERRIDE:
    validation_summary_candidates.append(Path(VALIDATION_SUMMARY_PATH_OVERRIDE))
validation_summary_candidates.append(
    REPO_ROOT / "supplementary" / "failure_mode_validation_h100" / "failure_mode_validation" / "tables" / "failure_mode_validation_summary.json"
)
if USE_GOOGLE_DRIVE:
    validation_summary_candidates.append(
        DRIVE_MOUNT_POINT / Path("MyDrive/SpectralBioFailureModeValidation/failure_mode_validation/tables/failure_mode_validation_summary.json")
    )
validation_summary_candidates += sorted(REPO_ROOT.glob("supplementary/**/failure_mode_validation_summary.json"))
validation_summary_path = None
validation_summary = None
for candidate in validation_summary_candidates:
    resolved = resolve_existing_path(candidate)
    if resolved.exists():
        validation_summary_path = resolved
        validation_summary = json.loads(validation_summary_path.read_text(encoding="utf-8"))
        break

candidate_modes = pd.read_csv(screen_summary_path.parent / "candidate_failure_modes.csv")
validation_pool = pd.read_csv(screen_summary_path.parent / "candidate_validation_pool.csv")
failure_examples = None
if validation_summary_path is not None:
    examples_path = resolve_existing_path(validation_summary_path.parent / "failure_mode_examples.csv")
    if examples_path.exists():
        failure_examples = pd.read_csv(examples_path)

print("screen_summary_path =", screen_summary_path)
print("validation_summary_path =", validation_summary_path)
print("selected_regime =", screen_summary["selected_regime"])
print("ACABEI PODE IR PARA O PR?XIMO")


In [ ]:
top_modes = candidate_modes.head(8).copy()
plt.figure(figsize=(10, 5))
plt.barh(top_modes["regime"][::-1], top_modes["composite_score"][::-1], color="#2459a6")
plt.xlabel("Composite candidate score")
plt.title("Top failure-mode candidates")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "top_failure_mode_candidates.png", dpi=180, bbox_inches="tight")
plt.close()

if failure_examples is not None and not failure_examples.empty:
    subset = failure_examples.sort_values("frob_norm_650m", ascending=False).head(20).copy()
    plt.figure(figsize=(8, 5))
    plt.bar(range(len(subset)), subset["frob_norm_650m"], color="#b42318")
    plt.xticks(range(len(subset)), subset["gene"] + ":" + subset["validation_role"], rotation=80)
    plt.ylabel("650M covariance norm")
    plt.title("Highest-signal validation examples")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "validation_examples_top20.png", dpi=180, bbox_inches="tight")
    plt.close()

candidate_modes.to_csv(TABLES_DIR / "candidate_failure_modes.csv", index=False)
validation_pool.to_csv(TABLES_DIR / "candidate_validation_pool.csv", index=False)
if failure_examples is not None:
    failure_examples.to_csv(TABLES_DIR / "failure_mode_examples.csv", index=False)

print("Packaging figures and tables copied.")
print("ACABEI PODE IR PARA O PR?XIMO")


In [ ]:
selected_regime = str(screen_summary["selected_regime"])
ready_for_h100 = bool(screen_summary["ready_for_h100"])
validation_ready_for_abstract = bool(validation_summary and validation_summary.get("executed_heavy_stage") and validation_summary.get("validated"))

if validation_ready_for_abstract:
    abstract_patch = (
        f"We further identify a structured failure mode centered on `{selected_regime}` variants: "
        "under a stronger 650M backbone, shortlisted candidate positives retain elevated covariance displacement "
        "relative to matched pathogenic controls despite weaker separation by likelihood alone. "
        "This result supports a narrower practical claim that local covariance can expose a perturbation regime "
        "that likelihood-centric scoring underweights."
    )
    content_patch = (
        f"A focused stronger-backbone validation of the `{selected_regime}` shortlist supports the screening result. "
        "When the candidate variants are re-scored with ESM2-650M, covariance-based separation remains cleaner than "
        "likelihood-only separation against matched pathogenic controls, consistent with a structured representational "
        "failure mode rather than a purely benchmark-specific fluctuation."
    )
    readiness = "Strong enough to mention in both abstract and content."
elif ready_for_h100:
    abstract_patch = (
        "The T4 failure-mode screen identified an interpretable candidate regime, but the H100 validation did not yet "
        "clear the threshold for an abstract-level claim. The result is better treated as a content-only exploratory addendum."
    )
    content_patch = (
        f"The `{selected_regime}` regime emerged as the strongest structured disagreement pattern in the screen, but its "
        "stronger-backbone validation remains suggestive rather than definitive. It can be reported as an exploratory "
        "failure-mode analysis without changing the main bounded claim of the paper."
    )
    readiness = "Content-only if you want to stay conservative."
else:
    abstract_patch = "No failure-mode result is currently strong enough to alter the abstract. The benchmark story should remain unchanged."
    content_patch = (
        f"The `{selected_regime}` screen is still useful internally, but it does not yet justify a new headline scientific claim. "
        "Keep the paper centered on bounded, gene-dependent covariance signal rather than on model failure discovery."
    )
    readiness = "Do not change the abstract; keep the current paper narrative."

readiness_md = f"# Submission Readiness\n\nSelected regime: `{selected_regime}`\n\nReady for H100: `{ready_for_h100}`\n\nValidation strong enough for abstract: `{validation_ready_for_abstract}`\n\nRecommendation:\n{readiness}\n"

(TEXT_DIR / "abstract_patch.md").write_text(abstract_patch + "\n", encoding="utf-8")
(TEXT_DIR / "content_patch.md").write_text(content_patch + "\n", encoding="utf-8")
(TEXT_DIR / "submission_readiness.md").write_text(readiness_md, encoding="utf-8")
(TEXT_DIR / "packaging_summary.json").write_text(json.dumps({
    "selected_regime": selected_regime,
    "ready_for_h100": ready_for_h100,
    "validation_ready_for_abstract": validation_ready_for_abstract,
    "recommendation": readiness,
}, indent=2, ensure_ascii=False), encoding="utf-8")

print(readiness_md)
print("ACABEI PODE IR PARA O PR?XIMO")


In [ ]:
import shutil

ZIP_BASE = ARTIFACT_ROOT / "failure_mode_packaging_bundle"
zip_path = Path(shutil.make_archive(str(ZIP_BASE), "zip", ARTIFACT_ROOT))
print("Bundle written to", zip_path)
display(FileLink(str(zip_path)))

try:
    from google.colab import files

    files.download(str(zip_path))
    print("Auto-download requested through google.colab.files.download.")
except Exception as error:
    print("Auto-download was not triggered:", error)

print("ACABEI PODE IR PARA O PR?XIMO")
